In [1]:
import pandas as pd
import matplotlib.pyplot as plt


In [4]:
pre = pd.read_csv(r"data\pre.csv")
recall = pd.read_csv(r"data\recall.csv")
speed = pd.read_csv(r"data\speed.csv")
correct_recall = pd.read_csv(r"data\answer_recall.csv")
correct_speed = pd.read_csv(r"data\answer_speed.csv")
overlap = pd.read_csv(r"data\overlap.csv")

In [ ]:
def compare_answers(responses_df, answer_key_df, section_name):
    responses = responses_df.copy()
    responses.columns = responses.columns.str.strip()

    answer_key = answer_key_df.copy()
    answer_key.columns = answer_key.columns.str.strip()

    response_q_cols = [c for c in responses.columns if c.isdigit()]
    key_q_cols = [c for c in answer_key.columns if c.isdigit()]
    common_q_cols = [c for c in response_q_cols if c in key_q_cols]

    if not common_q_cols:
        raise ValueError(f"No matching question columns found for {section_name}.")

    # Convert to numeric; non-numeric text (e.g. "1:00 remaining") becomes NaN.
    responses[common_q_cols] = responses[common_q_cols].apply(pd.to_numeric, errors="coerce")
    key_series = pd.to_numeric(answer_key.loc[0, common_q_cols], errors="coerce")

    # Ignore questions where the answer key is missing.
    valid_q_cols = [q for q in common_q_cols if pd.notna(key_series[q])]

    correctness = responses[valid_q_cols].eq(key_series[valid_q_cols], axis=1)

    first = responses.get("First Name", "").fillna("").astype(str).str.strip()
    last = responses.get("Last Name", "").fillna("").astype(str).str.strip()
    person = (first + " " + last).str.replace(r"\s+", " ", regex=True).str.strip()
    person = person.mask(person.eq(""), "Unknown")

    # right/wrong totals per person.
    summary = pd.DataFrame({
        "Person": person,
        "Document": responses.get("Document"),
        "Right": correctness.sum(axis=1),
        "Wrong": len(valid_q_cols) - correctness.sum(axis=1),
        "Total_Compared": len(valid_q_cols)
    })

    detail = pd.concat(
        [responses[["First Name", "Last Name", "Document"]].reset_index(drop=True), correctness.reset_index(drop=True)],
        axis=1
    )

    print(f"\n{section_name.upper()} SUMMARY")
    display(summary.sort_values(["Right", "Wrong"], ascending=[False, True]).reset_index(drop=True))

    print(f"{section_name.upper()} QUESTION-BY-QUESTION (True=right, False=wrong)")
    display(detail)

    return summary, detail


recall_summary, recall_detail = compare_answers(recall, correct_recall, "recall")
speed_summary, speed_detail = compare_answers(speed, correct_speed, "speed")


RECALL SUMMARY


,Person,Document,Right,Wrong,Total_Compared
0,Nithin Sivakumaran Sivakumaran,B,11,9,20
1,Lakshmi Male,B,9,11,20
2,Maya McPortland,A,8,12,20
3,Krisha Malkan,A,8,12,20
4,Disha Math,B,7,13,20
5,Yashu Singhai,B,6,14,20
6,Jiayi Liu,A,6,14,20
7,Benjamin Edwards,A,5,15,20
8,Prithika Roy,A,5,15,20
9,Lauren,B,5,15,20


RECALL QUESTION-BY-QUESTION (True=right, False=wrong)


,First Name,Last Name,Document,1,2,3,4,5,6,7,...,11,12,13,14,15,16,17,18,19,20
0,Maya,McPortland,A,True,True,False,True,True,False,False,...,True,False,False,False,False,False,False,False,False,False
1,Yashu,Singhai,B,True,True,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False
2,Benjamin,Edwards,A,False,True,False,True,False,False,True,...,False,False,False,False,False,False,False,True,False,False
3,Alyssa,Luas,B,False,False,False,True,False,True,False,...,False,False,False,False,True,False,False,True,False,False
4,Prithika,Roy,A,False,False,False,True,False,True,False,...,False,True,False,False,False,False,True,False,False,True
5,Disha,Math,B,False,False,False,True,True,False,True,...,False,True,False,False,False,False,False,False,False,False
6,Lakshmi,Male,B,True,False,False,False,True,True,True,...,False,True,False,False,False,False,True,False,False,False
7,Jiayi,Liu,A,False,False,False,True,True,False,True,...,False,False,False,False,False,False,True,False,False,False
8,Krisha,Malkan,A,True,True,True,True,True,True,True,...,False,False,False,False,False,False,True,False,False,False
9,Kristina,Valikhovskaya,A,False,False,False,True,False,False,True,...,False,False,False,False,False,False,True,False,False,False



SPEED SUMMARY


,Person,Document,Right,Wrong,Total_Compared
0,Disha Math,B,9,10,19
1,Lauren,B,8,11,19
2,Lakshmi Male,B,7,12,19
3,Maya McPortland,A,6,13,19
4,Benjamin Edwards,A,6,13,19
5,Jiayi Liu,A,6,13,19
6,Yashu Singhai,B,5,14,19
7,Prithika Roy,A,5,14,19
8,Kristina Valikhovskaya,A,5,14,19
9,Krisha Malkan,A,5,14,19


SPEED QUESTION-BY-QUESTION (True=right, False=wrong)


,First Name,Last Name,Document,1,2,3,4,5,6,7,...,10,11,12,14,15,16,17,18,19,20
0,Maya,McPortland,A,False,True,True,True,True,True,False,...,False,False,False,False,False,False,False,False,False,False
1,Yashu,Singhai,B,True,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,Benjamin,Edwards,A,False,False,True,True,True,True,False,...,True,False,False,False,False,False,False,False,False,False
3,Alyssa,Luas,B,True,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,Disha,Math,B,True,True,False,True,True,True,True,...,True,False,False,False,False,False,False,False,False,False
5,Prithika,Roy,A,False,True,True,True,True,True,False,...,False,False,False,False,False,False,False,False,False,False
6,Kristina,Valikhovskaya,A,False,True,True,True,True,True,False,...,False,False,False,False,False,False,False,False,False,False
7,Krisha,Malkan,A,False,True,False,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False
8,Lauren,NaN,B,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False
9,Nithin,Sivakumaran,B,True,False,False,True,True,True,False,...,False,False,False,False,False,False,False,False,False,False
